# PART 05 · 2장 작업형 2 핵심 패턴 — 연습 문제 정답

교재 448쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

multi_path = 'data/part03_type2/multiclass_ticket_priority/'
multi_train = pd.read_csv(multi_path + 'train.csv')
multi_test = pd.read_csv(multi_path + 'test.csv')
multi_sample = pd.read_csv(multi_path + 'sample_submission.csv')


def prepare_features(reference_X, other_X):
    reference_X = reference_X.copy()
    other_X = other_X.copy()
    cat_cols = reference_X.select_dtypes(include='object').columns
    num_cols = reference_X.select_dtypes(exclude='object').columns
    for col in num_cols:
        median_value = reference_X[col].median()
        reference_X[col] = reference_X[col].fillna(median_value)
        other_X[col] = other_X[col].fillna(median_value)
    for col in cat_cols:
        reference_X[col] = reference_X[col].fillna('unknown')
        other_X[col] = other_X[col].fillna('unknown')
    reference_X = pd.get_dummies(reference_X)
    other_X = pd.get_dummies(other_X)
    other_X = other_X.reindex(columns=reference_X.columns, fill_value=0)
    return reference_X, other_X


print(multi_train.shape, multi_test.shape)

(780, 15) (260, 14)


## 문제 01

다중분류 검증 데이터에서 다음 두 RandomForestClassifier의 Macro F1을 비교하시오. A는 n_estimators=200, B는 n_estimators=300, min_samples_leaf=4, class_weight='balanced'이다. 점수가 높은 설정으로 전체 train을 다시 학습한 뒤 test 예측 클래스별 개수를 구하시오.

**정답 코드**

In [2]:
multi_target = 'priority_level'
multi_id = 'ticket_id'
multi_X_raw = multi_train.drop(
    columns=[multi_target, multi_id]
).copy()
multi_y = multi_train[multi_target].copy()
multi_test_raw = multi_test.drop(columns=[multi_id]).copy()

(
    multi_X_train_raw,
    multi_X_valid_raw,
    multi_y_train,
    multi_y_valid
) = train_test_split(
    multi_X_raw,
    multi_y,
    test_size=0.2,
    random_state=0,
    stratify=multi_y
)

multi_X_train, multi_X_valid = prepare_features(
    multi_X_train_raw, multi_X_valid_raw
)

model_a = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
model_b = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=0
)

model_a.fit(multi_X_train, multi_y_train)
model_b.fit(multi_X_train, multi_y_train)

score_a = f1_score(
    multi_y_valid,
    model_a.predict(multi_X_valid),
    average='macro'
)
score_b = f1_score(
    multi_y_valid,
    model_b.predict(multi_X_valid),
    average='macro'
)
print('score_A:', round(score_a, 4))
print('score_B:', round(score_b, 4))

if score_a >= score_b:
    best_setting = 'A'
    best_model = RandomForestClassifier(
        n_estimators=200,
        random_state=0
    )
else:
    best_setting = 'B'
    best_model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=4,
        class_weight='balanced',
        random_state=0
    )

multi_X_full, multi_test_encoded = prepare_features(
    multi_X_raw, multi_test_raw
)
best_model.fit(multi_X_full, multi_y)
multi_test_pred = best_model.predict(multi_test_encoded)

print('best_setting:', best_setting)
print(pd.Series(multi_test_pred).value_counts().sort_index())

score_A: 0.4693
score_B: 0.5899


best_setting: B
0    112
1    116
2     32
Name: count, dtype: int64


**간단 해설**

두 설정의 Macro F1을 비교한 뒤 조건문으로 더 높은 설정을 선택한다. 동률이면 A를 선택한다. 이번 결과에서는 B의 점수가 더 높아 B 설정으로 전체 train을 다시 학습했으며, test 예측값은 클래스 0이 112개, 클래스 1이 116개, 클래스 2가 32개다.

## 문제 02

target이 delivery_minutes이고 평가 지표가 RMSE이다. 문제 유형과 제출할 값으로 가장 적절한 것은 무엇인가?

**간단 해설**

delivery_minutes는 연속형 수치이고 RMSE는 회귀 오차 지표다. 따라서 회귀 모형의 predict() 결과를 제출한다.

---

**정답** 1. best_setting은 B이며 test 예측은 클래스 0이 112개, 클래스 1이 116개, 클래스 2가 32개다, 2. 회귀 문제이므로 predict( ) 결과인 연속형 예측값을 제출한다.